# Univariate Analysis — Optimized

This notebook performs univariate exploratory data analysis (EDA) on the
Placement dataset: central tendency, percentiles, IQR-based outlier
detection/treatment, frequency tables, skewness/kurtosis, and
variance/standard deviation.

**What changed from the original version:**
- Fixed a critical bug where outlier capping used the wrong comparison
  operator and silently overwrote almost the entire column.
- Replaced unsafe chained indexing (`df[col][mask] = val`, which raises
  `SettingWithCopyWarning` and can fail silently) with `.loc[]`.
- Built one `Univariate()` function instead of four growing redefinitions —
  computed via `.describe()` and `Series` methods directly instead of
  repeated scalar-by-scalar assignment.
- Fixed `freqTable()` to use the column's actual count instead of a
  hardcoded `103` (which was only correct for `ssc_p`).
- Removed dead code (`dir(...)` dumps, commented-out lines, duplicate
  re-displays).
- `np.percentile` → `np.nanpercentile` so columns with missing values
  (like `salary`) compute correctly.


## 1. Load data

In [1]:
import pandas as pd
import numpy as np

dataset = pd.read_csv("Placement.csv")
dataset.head()

,sl_no,gender,ssc_p,ssc_b,hsc_p,hsc_b,hsc_s,degree_p,degree_t,workex,etest_p,specialisation,mba_p,status,salary
0,1,M,67.00,Others,91.00,Others,Commerce,58.00,Sci&Tech,No,55.0,Mkt&HR,58.80,Placed,270000.0
1,2,M,79.33,Central,78.33,Others,Science,77.48,Sci&Tech,Yes,86.5,Mkt&Fin,66.28,Placed,200000.0
2,3,M,65.00,Central,68.00,Central,Arts,64.00,Comm&Mgmt,No,75.0,Mkt&Fin,57.80,Placed,250000.0
3,4,M,56.00,Central,52.00,Central,Science,52.00,Sci&Tech,No,66.0,Mkt&HR,59.43,Not Placed,NaN
4,5,M,85.80,Central,73.60,Central,Commerce,73.30,Comm&Mgmt,No,96.8,Mkt&Fin,55.50,Placed,425000.0


In [2]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 215 entries, 0 to 214
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sl_no           215 non-null    int64  
 1   gender          215 non-null    object 
 2   ssc_p           215 non-null    float64
 3   ssc_b           215 non-null    object 
 4   hsc_p           215 non-null    float64
 5   hsc_b           215 non-null    object 
 6   hsc_s           215 non-null    object 
 7   degree_p        215 non-null    float64
 8   degree_t        215 non-null    object 
 9   workex          215 non-null    object 
 10  etest_p         215 non-null    float64
 11  specialisation  215 non-null    object 
 12  mba_p           215 non-null    float64
 13  status          215 non-null    object 
 14  salary          148 non-null    float64
dtypes: float64(6), int64(1), object(8)
memory usage: 25.3+ KB


## 2. Quantitative vs Qualitative columns

A column is **quantitative** if it's numeric, **qualitative** if it's an
`object` (string) dtype. One function, reused everywhere below — no need
to redefine it per-cell.

In [3]:
def quan_qual(df):
    """Split columns into quantitative (numeric) and qualitative (text) lists."""
    quan = df.select_dtypes(include="number").columns.tolist()
    qual = [c for c in df.columns if c not in quan]
    return quan, qual

quan, qual = quan_qual(dataset)
print("Quantitative:", quan)
print("Qualitative :", qual)

Quantitative: ['sl_no', 'ssc_p', 'hsc_p', 'degree_p', 'etest_p', 'mba_p', 'salary']
Qualitative : ['gender', 'ssc_b', 'hsc_b', 'hsc_s', 'degree_t', 'workex', 'specialisation', 'status']


## 3. Central tendency (quick look)

Mean, median, and mode of a single column, just to sanity-check before
building the full summary table.

In [4]:
dataset["ssc_p"].mean(), dataset["ssc_p"].median(), dataset["ssc_p"].mode()[0]

(67.30339534883721, 67.0, 62.0)

## 4. The `Univariate()` summary table

Builds one table per quantitative column with:
`Mean, Median, Mode, Q1, Q2, Q3, 99%, Max, IQR, 1.5×IQR rule, Lesser/Greater
fences, Min, Max, Skew, Kurtosis, Variance, Std`.

**Optimization:** the original called `dataset.describe()` inside the loop
(recomputing every quantile for every column on every iteration) and wrote
one scalar at a time via `descriptive[col]["row"] = value`. Here we compute
each column's stats once with `.describe()` + `Series` methods and assign
the whole column in one shot — far fewer redundant computations and no
repeated DataFrame-wide `.describe()` calls.

In [5]:
def Univariate(df, quan_cols):
    """Build a descriptive-statistics table for each quantitative column."""
    rows = ["Mean", "Median", "Mode", "Q1:25%", "Q2:50%", "Q3:75%", "99%",
            "Q4:100%", "IQR", "1.5rule", "Lesser", "Greater", "Min", "Max",
            "Skew", "Kurtosis", "Var", "Std"]
    descriptive = pd.DataFrame(index=rows, columns=quan_cols, dtype=float)

    for col in quan_cols:
        series = df[col]
        q1, q2, q3 = series.quantile([0.25, 0.50, 0.75])
        p99 = np.nanpercentile(series, 99)
        iqr = q3 - q1
        fence = 1.5 * iqr

        descriptive[col] = [
            series.mean(), series.median(), series.mode().iloc[0],
            q1, q2, q3, p99, series.max(),
            iqr, fence, q1 - fence, q3 + fence,
            series.min(), series.max(),
            series.skew(), series.kurtosis(),
            series.var(), series.std(),
        ]
    return descriptive

descriptive = Univariate(dataset, quan)
descriptive

,sl_no,ssc_p,hsc_p,degree_p,etest_p,mba_p,salary
Mean,108.000000,67.303395,66.333163,66.370186,72.100558,62.278186,2.886554e+05
Median,108.000000,67.000000,65.000000,66.000000,71.000000,62.000000,2.650000e+05
Mode,1.000000,62.000000,63.000000,65.000000,60.000000,56.700000,3.000000e+05
Q1:25%,54.500000,60.600000,60.900000,61.000000,60.000000,57.945000,2.400000e+05
Q2:50%,108.000000,67.000000,65.000000,66.000000,71.000000,62.000000,2.650000e+05
Q3:75%,161.500000,75.700000,73.000000,72.000000,83.500000,66.255000,3.000000e+05
99%,212.860000,87.000000,91.860000,83.860000,97.000000,76.114200,6.712000e+05
Q4:100%,215.000000,89.400000,97.700000,91.000000,98.000000,77.890000,9.400000e+05
IQR,107.000000,15.100000,12.100000,11.000000,23.500000,8.310000,6.000000e+04
1.5rule,160.500000,22.650000,18.150000,16.500000,35.250000,12.465000,9.000000e+04


## 5. Identify columns with outliers

A column has outliers on the low side if its actual `Min` is below the
`Lesser` fence, and on the high side if its actual `Max` is above the
`Greater` fence.

In [6]:
lesser_outlier_cols = [c for c in quan if descriptive[c]["Min"] < descriptive[c]["Lesser"]]
greater_outlier_cols = [c for c in quan if descriptive[c]["Max"] > descriptive[c]["Greater"]]

print("Columns with low-side outliers :", lesser_outlier_cols)
print("Columns with high-side outliers:", greater_outlier_cols)

Columns with low-side outliers : ['hsc_p']
Columns with high-side outliers: ['hsc_p', 'degree_p', 'salary']


## 6. Cap (winsorize) the outliers

**Bug fix:** the original used `<` for *both* the lower and upper cap, e.g.

```python
dataset[columnName][dataset[columnName] < descriptive[columnName]["Greater"]] = descriptive[columnName]["Greater"]
```

This caps everything *below* the Greater fence — i.e. nearly the whole
column — instead of everything *above* it. That's why every later summary
table collapsed (IQR became 0, skew/kurtosis went wild). The fix below also
uses `.loc[]` instead of chained indexing to avoid `SettingWithCopyWarning`
and guarantee the write actually lands on `dataset`.

In [7]:
dataset_capped = dataset.copy()

for col in lesser_outlier_cols:
    lesser = descriptive[col]["Lesser"]
    dataset_capped.loc[dataset_capped[col] < lesser, col] = lesser

for col in greater_outlier_cols:
    greater = descriptive[col]["Greater"]
    dataset_capped.loc[dataset_capped[col] > greater, col] = greater

# Recompute the summary table on the capped data to confirm outliers are gone
Univariate(dataset_capped, quan)

,sl_no,ssc_p,hsc_p,degree_p,etest_p,mba_p,salary
Mean,108.000000,67.303395,66.334744,66.358558,72.100558,62.278186,2.776486e+05
Median,108.000000,67.000000,65.000000,66.000000,71.000000,62.000000,2.650000e+05
Mode,1.000000,62.000000,63.000000,65.000000,60.000000,56.700000,3.000000e+05
Q1:25%,54.500000,60.600000,60.900000,61.000000,60.000000,57.945000,2.400000e+05
Q2:50%,108.000000,67.000000,65.000000,66.000000,71.000000,62.000000,2.650000e+05
Q3:75%,161.500000,75.700000,73.000000,72.000000,83.500000,66.255000,3.000000e+05
99%,212.860000,87.000000,91.129000,83.860000,97.000000,76.114200,3.900000e+05
Q4:100%,215.000000,89.400000,91.150000,88.500000,98.000000,77.890000,3.900000e+05
IQR,107.000000,15.100000,12.100000,11.000000,23.500000,8.310000,6.000000e+04
1.5rule,160.500000,22.650000,18.150000,16.500000,35.250000,12.465000,9.000000e+04


## 7. Frequency, relative frequency & cumulative frequency

**Bug fix:** the original hardcoded the divisor `103` (the unique-value
count for `ssc_p` specifically), which silently produced wrong relative
frequencies — and cumulative sums above 1 — for every other column (you can
see `Cusum` reaching 2.09 in the original `ssc_p` table, and topping 1.4
for `salary`). Using `len(df)` (or `series.count()` if you want to exclude
NaNs) fixes this for any column.

In [8]:
def freq_table(col, df):
    """Frequency, relative frequency, and cumulative relative frequency for one column."""
    counts = df[col].value_counts()
    n = df[col].count()  # non-null count, correct denominator for any column
    table = pd.DataFrame({
        "Unique_Values": counts.index,
        "Frequency": counts.values,
    })
    table["Relative Frequency"] = table["Frequency"] / n
    table["Cusum"] = table["Relative Frequency"].cumsum()
    return table

freq_table("ssc_p", dataset)

,Unique_Values,Frequency,Relative Frequency,Cusum
0,62.00,11,0.051163,0.051163
1,63.00,10,0.046512,0.097674
2,67.00,9,0.041860,0.139535
3,52.00,9,0.041860,0.181395
4,73.00,9,0.041860,0.223256
...,...,...,...,...
98,69.70,1,0.004651,0.981395
99,80.92,1,0.004651,0.986047
100,83.00,1,0.004651,0.990698
101,86.50,1,0.004651,0.995349


In [9]:
freq_table("salary", dataset)

,Unique_Values,Frequency,Relative Frequency,Cusum
0,300000.0,22,0.148649,0.148649
1,250000.0,18,0.121622,0.270270
2,240000.0,15,0.101351,0.371622
3,260000.0,7,0.047297,0.418919
4,360000.0,6,0.040541,0.459459
5,200000.0,6,0.040541,0.500000
6,265000.0,6,0.040541,0.540541
7,220000.0,5,0.033784,0.574324
8,275000.0,5,0.033784,0.608108
9,210000.0,4,0.027027,0.635135


## 8. Final summary table (on the original, uncapped data)

For reference — the full `Mean … Std` table computed on the original
dataset (not the outlier-capped version), matching what the notebook
originally intended to show.

In [10]:
Univariate(dataset, quan)

,sl_no,ssc_p,hsc_p,degree_p,etest_p,mba_p,salary
Mean,108.000000,67.303395,66.333163,66.370186,72.100558,62.278186,2.886554e+05
Median,108.000000,67.000000,65.000000,66.000000,71.000000,62.000000,2.650000e+05
Mode,1.000000,62.000000,63.000000,65.000000,60.000000,56.700000,3.000000e+05
Q1:25%,54.500000,60.600000,60.900000,61.000000,60.000000,57.945000,2.400000e+05
Q2:50%,108.000000,67.000000,65.000000,66.000000,71.000000,62.000000,2.650000e+05
Q3:75%,161.500000,75.700000,73.000000,72.000000,83.500000,66.255000,3.000000e+05
99%,212.860000,87.000000,91.860000,83.860000,97.000000,76.114200,6.712000e+05
Q4:100%,215.000000,89.400000,97.700000,91.000000,98.000000,77.890000,9.400000e+05
IQR,107.000000,15.100000,12.100000,11.000000,23.500000,8.310000,6.000000e+04
1.5rule,160.500000,22.650000,18.150000,16.500000,35.250000,12.465000,9.000000e+04
